In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#  Create flag Parameter

In [0]:
dbutils.widgets.text("Incremental_flag","0")

In [0]:
incremental_flag=dbutils.widgets.get("Incremental_flag")
print(incremental_flag)

0


# Creatiing Dimension Model

In [0]:
df_src=spark.sql(''' SELECT DISTINCT(Model_ID) as Model_ID,Category
FROM parquet.`abfss://silver@carprojectazurestorage.dfs.core.windows.net/carsales`''')


In [0]:
df_src.display()

Model_ID,Category
Hon-M219,Hon
Hyu-M161,Hyu
Toy-M101,Toy
Jee-M11,Jee
Tat-M192,Tat
Toy-M105,Toy
For-M16,For
Aud-M228,Aud
Nis-M81,Nis
Toy-M204,Toy


## dim_sink - Initial and Incremental

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    df_sink=spark.sql('''select dim_model_key,Model_ID,Category 
                  from cars_catalog.gold.dim_model
                  ''')
else:
    df_sink=spark.sql('''select 1 as dim_model_key,Model_ID,Category 
                  from parquet.`abfss://silver@carprojectazurestorage.dfs.core.windows.net/carsales`
                  where 1=0''')
            
df_sink.display()

dim_model_key,Model_ID,Category


## filter records

In [0]:
df_filter=df_src.join(df_sink,df_src.Model_ID==df_sink.Model_ID,'left').select(df_src.Model_ID,df_src.Category,df_sink.dim_model_key)
df_filter.display()

Model_ID,Category,dim_model_key
Hon-M219,Hon,null
Hyu-M161,Hyu,null
Toy-M101,Toy,null
Jee-M11,Jee,null
Tat-M192,Tat,null
Toy-M105,Toy,null
For-M16,For,null
Aud-M228,Aud,null
Nis-M81,Nis,null
Toy-M204,Toy,null


## Filtering new and old records

### old records

In [0]:
df_filter_old=df_filter.filter(col('dim_model_key').isNotNull())
df_filter_old.display()

Model_ID,Category,dim_model_key


### new records

In [0]:
df_filter_new=df_filter.filter(col('dim_model_key').isNull()).select(df_src['Model_ID'],df_src['Category'])
df_filter_new.display()

Model_ID,Category
Hon-M219,Hon
Hyu-M161,Hyu
Toy-M101,Toy
Jee-M11,Jee
Tat-M192,Tat
Toy-M105,Toy
For-M16,For
Aud-M228,Aud
Nis-M81,Nis
Toy-M204,Toy


# Creating Surrogate Key

**fetch the max surrogate key from existing table**

In [0]:
if incremental_flag=='0':
    max_value=1
else:
    max_value=spark.sql('''select max(dim_model_key) from cars_catalog.gold.dim_model''')
    max_value=max_value.collect()[0][0]
print(max_value)

1


**create the surrogate key and add it**

In [0]:
df_filter_new=df_filter_new.withColumn('dim_model_key',monotonically_increasing_id()+max_value)

df_filter_new.display()

Model_ID,Category,dim_model_key
Hon-M219,Hon,1
Hyu-M161,Hyu,2
Toy-M101,Toy,3
Jee-M11,Jee,4
Tat-M192,Tat,5
Toy-M105,Toy,6
For-M16,For,7
Aud-M228,Aud,8
Nis-M81,Nis,9
Toy-M204,Toy,10


### Creating final Df - df_filter_old + df_filter_new

In [0]:
df_final=df_filter_old.union(df_filter_new)
df_final.display()

Model_ID,Category,dim_model_key
Hon-M219,Hon,1
Hyu-M161,Hyu,2
Toy-M101,Toy,3
Jee-M11,Jee,4
Tat-M192,Tat,5
Toy-M105,Toy,6
For-M16,For,7
Aud-M228,Aud,8
Nis-M81,Nis,9
Toy-M204,Toy,10


# SCD Type - I

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_model'):
    delta_table=DeltaTable.forPath(spark,"abfss://gold@carprojectazurestorage.dfs.core.windows.net/dim_model")
    delta_table .alias('dlt').merge(df_final.alias('s'),'dlt.dim_model_key=s.dim_model_key')\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
    print("incremenatl")
else:
    df_final.write.format('Delta')\
            .mode("overwrite")\
            .option('path','abfss://gold@carprojectazurestorage.dfs.core.windows.net/dim_model')\
            .saveAsTable('cars_catalog.gold.dim_model')

In [0]:
%sql
select * from cars_catalog.gold.dim_model

Model_ID,Category,dim_model_key
Hon-M219,Hon,1
Hyu-M161,Hyu,2
Toy-M101,Toy,3
Jee-M11,Jee,4
Tat-M192,Tat,5
Toy-M105,Toy,6
For-M16,For,7
Aud-M228,Aud,8
Nis-M81,Nis,9
Toy-M204,Toy,10
